# Entrenamiento del modelo de demanda:

En este notebook entrenaremos el modelo Tide de demanda, acompañado a este notebook hay otros 2 notebooks para las otras dos variables predecibles

Exclusivamente en este notebook se ejecutara un codigo que una los dos df, el de train y test y lo subira a minio de nuevo como dataframe_final

In [1]:
import torch
import torch.nn as nn
import time
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import sys
import os
from pathlib import Path

root = Path.cwd().parent.parent
sys.path.append(str(root))

from minio_utils import MinioSparkClient


from pyspark.ml.feature import StringIndexer, OneHotEncoder,VectorAssembler, StandardScaler, SQLTransformer, Imputer
from pyspark.sql.types import FloatType
from pyspark.sql import functions as F

from pyspark.ml.functions import vector_to_array
from pyspark.sql.window import Window
import math
from torch.utils.data import IterableDataset, DataLoader
import pyarrow.parquet as pq

from darts import TimeSeries
from darts.dataprocessing.transformers import InvertibleMapper, Mapper
from darts.dataprocessing.pipeline import Pipeline
from darts.dataprocessing.transformers import Scaler
from darts.models import TiDEModel
from darts.dataprocessing.transformers import MissingValuesFiller
from darts.utils.likelihood_models import QuantileRegression
from pytorch_lightning.callbacks import EarlyStopping
from sklearn.preprocessing import RobustScaler


from clearml import Task
from tqdm.auto import tqdm
import joblib
import pandas as pd
from darts import TimeSeries
from darts.utils.timeseries_generation import datetime_attribute_timeseries
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder as SklearnOneHotEncoder,StandardScaler as SklearnStandardScaler
from sklearn.cluster import KMeans


from setup import setenv
setenv()


The StatsForecast module could not be imported. To enable support for the AutoARIMA, AutoETS and Croston models, please consider installing it.


In [2]:

spark = MinioSparkClient(
    endpoint=os.getenv("MINIO_ENDPOINT", "").replace("http://", "").replace("https://", ""),
    access_key=os.getenv("MINIO_ACCESS_KEY"),
    secret_key=os.getenv("MINIO_SECRET_KEY"),
    bucket_name="pd2",
    base_dir="cityenjoyer",
    memory = 16,
    heapsize = 8,
    num_part = 2000,
    verbose=True
)
spark.connect()

https://mmlspark.azureedge.net/maven added as a remote repository with the name: repo-1
Ivy Default Cache set to: /home/QuiSioU/.ivy2/cache
The jars for the packages stored in: /home/QuiSioU/.ivy2/jars
com.microsoft.azure#synapseml_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-bf06b253-5d3c-4c1b-a67c-9aa4a754eab0;1.0
	confs: [default]
	found com.microsoft.azure#synapseml_2.12;1.1.2 in central
	found com.microsoft.azure#synapseml-core_2.12;1.1.2 in central
	found org.apache.spark#spark-avro_2.12;3.5.0 in central
	found org.tukaani#xz;1.9 in central
	found commons-lang#commons-lang;2.6 in central
	found org.scalactic#scalactic_2.12;3.2.14 in central


:: loading settings :: url = jar:file:/home/QuiSioU/UCM/y3/PD2/.venv/lib/python3.13/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


	found org.scala-lang#scala-reflect;2.12.15 in central
	found io.spray#spray-json_2.12;1.3.5 in central
	found com.jcraft#jsch;0.1.54 in central
	found org.apache.httpcomponents.client5#httpclient5;5.1.3 in central
	found org.apache.httpcomponents.core5#httpcore5;5.1.3 in central
	found org.apache.httpcomponents.core5#httpcore5-h2;5.1.3 in central
	found org.slf4j#slf4j-api;1.7.25 in central
	found commons-codec#commons-codec;1.15 in central
	found org.apache.httpcomponents#httpmime;4.5.13 in central
	found org.apache.httpcomponents#httpclient;4.5.13 in central
	found org.apache.httpcomponents#httpcore;4.4.13 in central
	found commons-logging#commons-logging;1.2 in central
	found com.linkedin.isolation-forest#isolation-forest_3.5.0_2.12;3.0.5 in central
	found com.chuusai#shapeless_2.12;2.3.10 in central
	found org.testng#testng;6.8.8 in central
	found org.beanshell#bsh;2.0b4 in central
	found com.beust#jcommander;1.27 in central
	found com.microsoft.azure#synapseml-deep-learning_2.12;

In [4]:
from datetime import timedelta
df1 = spark.read_parquet("prepared_for_model/20260323_132105_agg.parquet")
df2 = spark.read_parquet("prepared_for_model/20260405_154246_agg.parquet")

# 2. Obtener las fechas clave
# Usamos collect()[0][0] para pasar el valor de la fila/columna a una variable de Python
last_date_df1 = df1.select(F.max("timestamp")).collect()[0][0]
first_date_df2 = df2.select(F.min("timestamp")).collect()[0][0]

print(f"Última fecha DF1: {last_date_df1}")
print(f"Primera fecha DF2: {first_date_df2}")

# 3. Comprobar la continuidad (Diferencia de exactamente 1 día)
if last_date_df1 + timedelta(hours=1) == first_date_df2:
    print("✅ Continuidad confirmada. Concatenando...")

    df_final = df1.unionByName(df2)
    
    
    print(f"Total filas tras concatenar: {df_final.count()}")
else:
    diff = (first_date_df2 - last_date_df1).days
    print(f"❌ Error de continuidad: Hay un salto de {diff} días entre los datasets.")
   

Última fecha DF1: 2025-11-30 23:00:00
Primera fecha DF2: 2025-12-01 00:00:00
✅ Continuidad confirmada. Concatenando...


Total filas tras concatenar: 28916185


In [5]:
df_final.show(5)

+--------+------------+-------------------+------+------------+----------+---------+---------+----+---+------------------+--------------------+-------------------+-------------------+
|VendorID|PULocationID|          timestamp|demand|avg_distance|avg_amount| Latitude|Longitude|hour|dow|          hour_sin|            hour_cos|            dow_sin|            dow_cos|
+--------+------------+-------------------+------+------------+----------+---------+---------+----+---+------------------+--------------------+-------------------+-------------------+
|       0|         263|2023-06-01 04:00:00|    14|     10730.5| 3877.7144|40.778767|-73.95101|   4|  5|0.8660254037844386|  0.5000000000000001|-0.9749279121818236|-0.2225209339563146|
|       0|         262|2023-06-01 06:00:00|    59|    4567.576| 2454.5425|40.775932|-73.94651|   6|  5|               1.0|6.123233995736766...|-0.9749279121818236|-0.2225209339563146|
|       0|         163|2023-06-01 09:00:00|   141|   4182.0566| 2570.9148| 40.76

In [6]:
spark.write_parquet(df_final, "final_data.parquet")

Codigo en caso de que no esten en local los datos

In [7]:
# Limpiamos nulos
df_clean = df_final.na.drop(subset=["demand", "PULocationID", "VendorID", "timestamp"])

# Creamos una columna que sea el ID único de la serie temporal
df_clean = df_clean.withColumn(
    "Series_ID", 
    F.concat_ws("_", F.col("VendorID"), F.col("PULocationID"))
)

# Ahora particionamos por esa nueva columna, 
# esto nos permite tener un folder por cada combinación y 
# nos permite usar darts con mini datasets que podemos pasar a pandas
df_clean.write \
    .partitionBy("Series_ID") \
    .mode("overwrite") \
    .parquet("../data/final_data.parquet")

print("Datos exportados correctamente por Vendor y Zona.")

Datos exportados correctamente por Vendor y Zona.


## Preprocesamiento de los datos:

In [2]:
df_pd = pd.read_parquet("../data/final_data.parquet")

# Es vital que los datos estén ordenados temporalmente dentro de cada grupo
df_pd = df_pd.sort_values(by=["Series_ID", "timestamp"])

fecha_limite = df_pd['timestamp'].max() - pd.DateOffset(years=3)
df_pd = df_pd[df_pd['timestamp'] >= fecha_limite].copy()

# =============================================
# PREPROCESAMIENTO DE VARIABLES
# =============================================
# LATITUD Y LONGITUD (Escalado espacial)
scaler_espacial = MinMaxScaler()
cols_geo = ["Latitude", "Longitude"]
df_pd[cols_geo] = scaler_espacial.fit_transform(df_pd[cols_geo])
df_pd[cols_geo] = df_pd[cols_geo].astype("float32")

# VARIABLES NUMÉRICAS (Float32)
cols_numericas = ["demand", "avg_distance", "avg_amount"]
df_pd[cols_numericas] = df_pd[cols_numericas].astype("float32")

print("Aplicando One-Hot Encoding a VendorID...")
encoder_vendor = SklearnOneHotEncoder(sparse_output=False, drop='first', handle_unknown='ignore')
dummies_array = encoder_vendor.fit_transform(df_pd[['VendorID']])
nombres_dummies = encoder_vendor.get_feature_names_out(['VendorID'])
df_pd[nombres_dummies] = dummies_array
df_pd["VendorID"] = df_pd["VendorID"].astype("float32") # Mantenemos el original para Darts

columnas_estaticas = list(nombres_dummies) + ["Latitude", "Longitude"]
df_pd[columnas_estaticas] = df_pd[columnas_estaticas].astype("float32")
print(f"Variables estáticas que verá el modelo: {columnas_estaticas}")



print("\nIniciando Segmentación Basada en Características (Temporal Signatures, CV & ADI)...")

### Borrar aqui

# 1. Preparación de franjas horarias estrictas (Firmas Temporales)
# fecha_min = df_pd['timestamp'].min()
# fecha_max = df_pd['timestamp'].max()
# horas_totales = max(1, round((fecha_max - fecha_min).total_seconds() / 3600))

# df_pd['is_weekend'] = df_pd['timestamp'].dt.dayofweek >= 5


# # 1. Agrupación y cálculo del ADI
# agrupado = df_pd.groupby('Series_ID').agg(
#     total_amount=('avg_amount', 'sum'),
#     std_amount=('avg_amount', 'std'),
#     weekend_amount=('avg_amount', lambda x: x[df_pd.loc[x.index, 'is_weekend']].sum()),
#     horas_con_actividad=('avg_amount', 'count') 
# ).reset_index()

# agrupado['mean_real'] = agrupado['total_amount'] / horas_totales
# agrupado['cv'] = agrupado['std_amount'] / (agrupado['mean_real'] + 0.0001)

# # 🎯 CORRECCIÓN: Volvemos a calcular el weekend_ratio
# agrupado['weekend_ratio'] = agrupado['weekend_amount'] / (agrupado['total_amount'] + 0.0001)

# # 🎯 EL ÍNDICE ADI (Average Demand Interval)
# # Mide cuántas horas pasan entre viajes. Si horas_con_actividad es 0, ponemos infinito.
# agrupado['adi'] = np.where(agrupado['horas_con_actividad'] > 0, 
#                            horas_totales / agrupado['horas_con_actividad'], 
#                            np.inf)

# agrupado['cv2'] = agrupado['cv'] ** 2

# # Limpiamos NaNs por si alguna serie tuvo 0 varianza
# agrupado.fillna({'cv': 0, 'cv2': 0}, inplace=True)

# # 2. FILTRO BASELINE MATEMÁTICO (Basado en Syntetos-Boylan)
# UMBRAL_ADI = 1.5 
# UMBRAL_CV2 = 0.5

# # Es BASELINE si es "Lumpy" extremo (Muy intermitente Y muy errática)
# mask_lumpy = ((agrupado['adi'] > UMBRAL_ADI) & (agrupado['cv2'] > UMBRAL_CV2)) | (agrupado['mean_real'] < 0.5)

# zonas_muertas = agrupado[mask_lumpy].copy()
# zonas_vivas = agrupado[~mask_lumpy].copy()
# zonas_muertas['segmento'] = 'BASELINE'

# print(f"   - Zonas enviadas a BASELINE (Ruido Matemático / Lumpy): {len(zonas_muertas)}")

### Borrar aqui

# =============================================
# CONVERSIÓN A DARTS TIMESERIES
# =============================================
col_tiempo = "timestamp"
col_grupo = ["PULocationID", "VendorID"]

# CREAMOS EL CALENDARIO MAESTRO
print("\nGenerando Calendario Maestro de Covariables Futuras...")
HORAS_EXTRA = 48
inicio_global = df_pd['timestamp'].min()
fin_global = df_pd['timestamp'].max() + pd.Timedelta(hours=HORAS_EXTRA)

indice_global = pd.date_range(start=inicio_global, end=fin_global, freq='h')
cov_hora_global = datetime_attribute_timeseries(indice_global, attribute="hour", cyclic=True)
cov_dia_global = datetime_attribute_timeseries(indice_global, attribute="dayofweek", cyclic=True)
calendario_maestro = cov_hora_global.stack(cov_dia_global).astype("float32")

def preparar_dataset_darts(df_segmento, nombre_segmento):
    """Convierte un DataFrame de Pandas en las 3 listas de TimeSeries para TiDE"""
    if df_segmento.empty:
        return [], [], []
    
    targets = TimeSeries.from_group_dataframe(
        df_segmento, group_cols=col_grupo, time_col=col_tiempo, value_cols="avg_amount",
        static_cols=columnas_estaticas, fill_missing_dates=True, freq='h', fillna_value=0           
    )
    
    past_covs = TimeSeries.from_group_dataframe(
        df_segmento, group_cols=col_grupo, time_col=col_tiempo, value_cols=["avg_distance", "demand"],
        fill_missing_dates=True, freq='h', fillna_value=0           
    )
    
    future_covs = [calendario_maestro] * len(targets)
    return targets, past_covs, future_covs

print(f"\nPreparando tensores para TiDE...")
# Nota: Ignoramos df_baseline intencionadamente
targets, past, future = preparar_dataset_darts(df_pd, "Precio")
# targets, past, future = preparar_dataset_darts(df_clean, "Precio")

print("✅ Pipeline de datos completado.")

Aplicando One-Hot Encoding a VendorID...
Variables estáticas que verá el modelo: ['VendorID_1', 'VendorID_2', 'VendorID_3', 'Latitude', 'Longitude']

Iniciando Segmentación Basada en Características (Temporal Signatures, CV & ADI)...

Generando Calendario Maestro de Covariables Futuras...

Preparando tensores para TiDE...
✅ Pipeline de datos completado.


In [3]:
# ==========================================
# DIVISIÓN TRAIN/VAL Y TRANSFORMACIÓN (MODULAR)
# ==========================================
CORTE = pd.Timestamp("2026-2-14 00:00:00") 
VENTANA_TRAIN = 96 # 72h input + 24h output
VENTANA_VAL = 168   # Evaluamos solo 1 día para evitar el error de horizonte lejano

def preparar_entrenamiento(targets, past, future, nombre_segmento):
    """
    Filtra, divide y aplica las transformaciones.
    Construye las series de validación con el contexto histórico necesario (72h).
    """
    if not targets:
        return {}, {}, None, None

    print(f"\nDividiendo datos y aplicando pipelines para: {nombre_segmento}...")

    INPUT_CHUNK = 72
    OUTPUT_CHUNK = 24

    tr_tgt, val_tgt = [], []
    tr_pst, val_pst = [], []
    tr_fut, val_fut = [], []

    zonas_descartadas = 0

    for tgt, pst, fut in zip(targets, past, future):
        
        # 1. ESCUDO: ¿Hay datos a ambos lados del corte?
        if tgt.start_time() < CORTE and (tgt.end_time() - CORTE).days >= 1:
            
            # Cortamos por la fecha exacta SOLO para ver la longitud del train
            tr_t_temp, ts_t_temp = tgt.split_before(CORTE)
            
            # 2. ESCUDO: ¿Tiene el tamaño mínimo? (Train > 72, Val > 24)
            if len(tr_t_temp) >= INPUT_CHUNK and len(ts_t_temp) >= OUTPUT_CHUNK:
                
                # Obtenemos el índice numérico exacto del corte
                idx = len(tr_t_temp) 
                
                # TRAIN: Todo desde el inicio hasta el corte (como siempre)
                tr_tgt.append(tgt[:idx])
                tr_pst.append(pst[:idx])
                
                # VALIDACIÓN: 72h antes del corte + 24h después del corte = 96h
        
                val_tgt.append(tgt[idx - INPUT_CHUNK : idx + VENTANA_VAL])
                val_pst.append(pst[idx - INPUT_CHUNK : idx + VENTANA_VAL])

                #El futuro es todo
                tr_fut.append(fut) 
                val_fut.append(fut)
                
                
            else:
                zonas_descartadas += 1
        else:
            zonas_descartadas += 1

    print(f"{nombre_segmento}: {len(tr_tgt)} series válidas listas. Descartadas: {zonas_descartadas}")

    if not tr_tgt:
        return {}, {}, None,None

    # ==========================================
    # PIPELINES (Log1p)
    # ==========================================
    target_pipeline = Pipeline([InvertibleMapper(fn=np.log1p, inverse_fn=np.expm1)])
    past_pipeline = Pipeline([Mapper(fn=np.log1p)])


    dict_train = {
        "targets": target_pipeline.fit_transform(tr_tgt),
        "past": past_pipeline.fit_transform(tr_pst),
        "future": tr_fut 
    }
    
    dict_val = {
        "targets_real": val_tgt, 
        "targets": target_pipeline.transform(val_tgt),
        "past": past_pipeline.transform(val_pst),
        "future": val_fut
    }
    return dict_train, dict_val, target_pipeline, past_pipeline

# ==========================================
# EJECUTAR PARA NUESTROS SEGMENTOS
# ==========================================
train, val, pipe, pipe_past = preparar_entrenamiento(targets, past, future, "PRECIO")

print("\n¡Todos los datos están listos y empaquetados para entrenar!")


Dividiendo datos y aplicando pipelines para: PRECIO...
PRECIO: 990 series válidas listas. Descartadas: 48

¡Todos los datos están listos y empaquetados para entrenar!


## Entrenamiento del modelo

In [4]:

torch.set_float32_matmul_precision('high') 
# ==========================================
# ESPECIALISTA 1: ALTO PRECIO (El Modelo Pesado)
# ==========================================
if train["targets"]:
    print("\n" + "="*50)
    print("ENTRENANDO ESPECIALISTA: ALTO PRECIO")
    print("="*50)

    detenedor = EarlyStopping(
        monitor="val_loss",
        patience=3,
        min_delta=0.001,
        mode="min",
    )

    modelo = TiDEModel(
        input_chunk_length=72,   
        output_chunk_length=24,  
        
        # Arquitectura pesada para absorber picos gigantes
        num_encoder_layers=2,  
        num_decoder_layers=2,  
        decoder_output_dim=128,
        hidden_size=256,         
        dropout=0.15, 
        
        use_static_covariates=True, 
        use_reversible_instance_norm=True, 

        batch_size=1024,          
        n_epochs=40,              
        optimizer_kwargs={"lr": 1e-3}, 
        random_state=42,         
        
        likelihood=QuantileRegression(quantiles=[0.10, 0.50, 0.90]),
        
        pl_trainer_kwargs={
            "accelerator": "gpu",
            "devices": -1,
            "gradient_clip_val": 0.5, 
            "callbacks": [detenedor],
            "precision": "32-true"
        },
        
        # MLOps
        work_dir="amount", 
        model_name="precio",        
        save_checkpoints=True,            
        force_reset=True,   
    )

    modelo.fit(
        series=train["targets"],              
        past_covariates=train["past"],        
        future_covariates=train["future"],        
        val_series=val["targets"],            
        val_past_covariates=val["past"],      
        val_future_covariates=val["future"],      
        dataloader_kwargs={"num_workers": 8},
        verbose=True,                        
    )


ENTRENANDO ESPECIALISTA: ALTO PRECIO


number of `past_covariates` features is <= `temporal_width_past`, leading to feature expansion.number of covariates: 2, `temporal_width_past=4`.
number of `future_covariates` features is <= `temporal_width_future`, leading to feature expansion.number of covariates: 4, `temporal_width_future=4`.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━━┳━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃    ┃ Name                  ┃ Type             ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0  │ criterion             │ MSELoss          │      0 │ train │     0 │
│ 1  │ train_criterion       │ MSELoss          │      0 │ train │     0 │
│ 2  │ val_criterion         │ MSELoss          │      0 │ train │     0 │
│ 3  │ train_metrics         │ MetricCollection │      0 │ train │     0 │
│ 4  │ val_metrics           │ MetricCollection │      0 │ train │     0 │
│ 5  │ rin                   │ RINorm           │      2 │ train │     0 │
│ 6  │ past_cov_projection   │ _ResidualBlock   │  1.8 K │ train │     0 │
│ 7  │ future_cov_projection │ _ResidualBlock   │  2.3 K │ train │     0 │
│ 8  │ encoders              │ Sequential       │  648 K │ train │     0 │
│ 9  │ decoders              │ Sequential       │  5.0 M │ train │     0 │
│ 10 │ temporal_decoder      │ _ResidualBlock   │ 13.7 K │ train │     0 │
│ 11 │ lookback_skip         │ Linear           │  5.3 K │ train │     0 │
└────┴───────────────────────┴──────────────────┴────────┴───────┴───────┘

Trainable params: 5.7 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 5.7 M                                                                                                
Total estimated model params size (MB): 22                                                                         
Modules in train mode: 58                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/home/QuiSioU/UCM/y3/PD2/.venv/lib/python3.13/site-packages/pytorch_lightning/utilities/_pytree.py:21: 
`isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` 
instead.

# Evaluación del modelo

In [5]:
modelo = TiDEModel.load_from_checkpoint(model_name= "precio", work_dir="amount", best=True, weights_only = False)

A diferencia de el notebook de pruebas, el baseline no será una media movil, ya que perderiamos la potencia de predecir una ventana temporal que es en verdad el problema interesante a resolver. Las zonas baseline son muy exporádicas y con muchos ceros, presuponiendo por los análisis que en esas zonas ese patron es constante, la nueva baseline es una Seasonal Naive, o en pocas palabras, predecir que la forma de la curva es la misma cada 24 horas.

In [6]:
from darts.metrics import mae
import numpy as np

print("📊 Calculando MAE de Validación y MAE Global...\n")

# 🔥 AHORA SÍ: Los 4 segmentos integrados perfectamente
segmentos = [
    # (Nombre, Modelo, Datos_Validación, Pipeline_Inverso)
    ("Precio", modelo, val, pipe),   # Camino B: Deep Learning
]

errores_globales = []

for nombre, modelo, val_data, pipe in segmentos:
    print(f"⏳ Evaluando {nombre}...")
    errores_zona = []

    # =======================================================
    # 🛣️ CAMINO A: HEURÍSTICA BASELINE (Media de las 2h previas)
    # =======================================================
    if nombre == "BASELINE":
        for serie in val_data["targets"]:
            errores_dias = []
            inicio = 72
            horizonte = 24
            
            while inicio + horizonte <= len(serie):
                # 1. Miramos las 24 HORAS EXACTAMENTE ANTERIORES (Copiamos el día de ayer)
                curva_ayer = serie[inicio - 24 : inicio].values()
                
                # 2. Sacamos la realidad de las 24h de hoy
                realidad_hoy = serie[inicio : inicio + horizonte].values()
                
                # 3. Calculamos el MAE comparando hora a hora (10am de ayer vs 10am de hoy)
                error_dia = np.mean(np.abs(realidad_hoy - curva_ayer))
                errores_dias.append(error_dia)
                
                inicio += 24 
                
            if errores_dias: 
                mae_zona_total = np.mean(errores_dias)
                errores_zona.append(mae_zona_total)
                errores_globales.append(mae_zona_total)

    # =======================================================
    # 🛣️ CAMINO B: DEEP LEARNING (TiDE + QuantileLoss + Log1p)
    # =======================================================
    else:
        preds_tf_hist = modelo.historical_forecasts(
            series=val_data["targets"],
            past_covariates=val_data["past"],
            future_covariates=val_data["future"],
            start=72,                   
            forecast_horizon=24,        
            stride=24,                  
            retrain=False,              
            last_points_only=False,     
            num_samples=200             
        )
        
        # Deshacemos el logaritmo de los datos reales para comparar justamente
        val_real = pipe.inverse_transform(val_data["targets"])
        
        for i, lista_dias_zona in enumerate(preds_tf_hist):
            errores_dias = [] 
            
            for dia_pred_tf in lista_dias_zona:
                # Invertimos el logaritmo de la predicción y sacamos la mediana (P50)
                dia_pred_real = pipe.inverse_transform(dia_pred_tf)
                pred_p50 = dia_pred_real.quantile(0.50)
                
                # Buscamos la realidad correspondiente y medimos el error
                actual_real = val_real[i].slice_intersect(pred_p50)
                error_dia = mae(actual_real, pred_p50)
                errores_dias.append(error_dia)
                
            if errores_dias:
                mae_zona_total = np.mean(errores_dias)
                errores_zona.append(mae_zona_total)
                errores_globales.append(mae_zona_total)
            
    # Resultado Promedio del Segmento
    if errores_zona:
        mae_promedio = np.mean(errores_zona)
        print(f"   🎯 MAE [{nombre}]: {mae_promedio / 100:.2f} euros")
    else:
        print(f"   ⚠️ No se pudieron evaluar zonas para {nombre}.")

# ==========================================
# 🌍 CÁLCULO DEL MAE GLOBAL
# ==========================================
if errores_globales:
    mae_total_real = np.mean(errores_globales)
    print("\n" + "="*50)
    print(f"🏆 MAE GLOBAL DEL SISTEMA (NY COMPLETO): {mae_total_real / 100:.2f}euros ")
    print("="*50)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


📊 Calculando MAE de Validación y MAE Global...

⏳ Evaluando Precio...
   🎯 MAE [Precio]: 5.60 euros

🏆 MAE GLOBAL DEL SISTEMA (NY COMPLETO): 5.60euros 


# Guardamos todo

In [32]:
import json
from json import encoder
import os


CARPETA_ARTEFACTOS = "darts_models"
os.makedirs(CARPETA_ARTEFACTOS, exist_ok=True)

print("💾 Guardando transformadores de preprocesamiento...")

# Guardamos el One-Hot Encoder
joblib.dump(encoder_vendor, f"{CARPETA_ARTEFACTOS}/encoder_vendor.pkl")

# Guardamos el Scaler Espacial (Lat/Lon)
joblib.dump(scaler_espacial, f"{CARPETA_ARTEFACTOS}/scaler_espacial.pkl")

print("✅ Artefactos guardados con éxito.")

💾 Guardando transformadores de preprocesamiento...
✅ Artefactos guardados con éxito.
